In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import math

DATA = Path('../data')
OUT = Path('../outputs')
orders = pd.read_csv(DATA/'orders.csv', parse_dates=['order_timestamp'])
picks = pd.read_csv(DATA/'picks.csv', parse_dates=['start_time','finish_time'])
items = pd.read_csv(DATA/'items.csv')
workers = pd.read_csv(DATA/'workers.csv')
zones = pd.read_csv(DATA/'zones.csv')
print({ 'orders': orders.shape, 'picks': picks.shape, 'items': items.shape, 'workers': workers.shape, 'zones': zones.shape })


{'orders': (900, 4), 'picks': (2443, 8), 'items': (180, 4), 'workers': (40, 3), 'zones': (6, 3)}


## Data quality checks and cleaning

In [2]:
# Normalize labels and remove duplicate pick IDs
picks['zone'] = picks['zone'].astype(str).str.upper().str.replace(' ', '', regex=False).replace({'NAN': np.nan})
picks = picks.drop_duplicates(subset=['pick_id']).copy()
picks['pick_duration_min'] = (picks['finish_time'] - picks['start_time']).dt.total_seconds()/60
# Missing start/finish or non-positive durations cannot be used as observed durations.
picks = picks.dropna(subset=['start_time','finish_time','pick_duration_min'])
picks = picks[picks['pick_duration_min'] > 0].copy()
# Cap extreme positive duration at 99th percentile to reduce influence of rare outliers.
picks['pick_duration_min'] = picks['pick_duration_min'].clip(upper=picks['pick_duration_min'].quantile(.99))
workers['shift'] = workers['shift'].fillna('Unknown')
workers['experience_band'] = workers['experience_band'].fillna('Unknown')
clean = (picks.merge(orders, on='order_id', how='left')
         .merge(items, on='item_id', how='left')
         .merge(workers, on='picker_id', how='left')
         .merge(zones, on='zone', how='left'))
for c in ['zone_type','distance_band','category','size','storage_class','shift','experience_band']:
    clean[c] = clean[c].fillna('Unknown')
clean['order_wait_min'] = ((clean['start_time'] - clean['order_timestamp']).dt.total_seconds()/60).clip(lower=0)
clean['order_size_band'] = pd.cut(clean['item_count'], [0,3,7,13,100], labels=['Small (1-3)','Medium (4-7)','Large (8-13)','Very large (14+)'])
clean['start_hour'] = clean['start_time'].dt.hour
clean['delay_flag'] = (clean['pick_duration_min'] > clean['pick_duration_min'].median()).astype(int)
clean.to_csv(OUT/'cleaned_warehouse_picks.csv', index=False)
clean.head()


,pick_id,order_id,item_id,picker_id,zone,start_time,finish_time,quantity,pick_duration_min,item_count,...,size,storage_class,shift,experience_band,zone_type,distance_band,order_wait_min,order_size_band,start_hour,delay_flag
0,P000001,O00422,I0065,W027,A-01,2026-09-20 05:02:00,2026-09-20 05:15:37.674292,3,13.627905,1,...,Large,Heavy,Evening,Intermediate,Fast-pick,Short,47.0,Small (1-3),5,1
1,P000002,O00569,I0076,W006,C-02,2026-08-10 08:24:00,2026-08-10 08:34:49.027526,5,10.817125,7,...,Medium,Heavy,Morning,Experienced,Bulk,Long,13.0,Medium (4-7),8,0
2,P000003,O00361,I0090,W014,C-02,2026-09-18 23:55:00,2026-09-19 00:09:35.309728,6,14.588495,1,...,Medium,Ambient,Evening,Experienced,Bulk,Long,53.0,Small (1-3),23,1
3,P000004,O00049,I0133,W013,B-01,2026-09-15 15:22:00,2026-09-15 15:32:01.071955,4,10.017866,1,...,Small,Temperature-controlled,Night,Experienced,Standard,Medium,6.0,Small (1-3),15,0
4,P000005,O00340,I0085,W020,C-01,2026-08-09 05:01:00,2026-08-09 05:18:06.725573,5,17.112093,1,...,Small,Fragile,Morning,Intermediate,Bulk,Long,56.0,Small (1-3),5,1


## EDA

In [3]:
zone_summary = clean.groupby('zone').agg(pick_count=('pick_id','count'), avg_pick_time_min=('pick_duration_min','mean'), median_pick_time_min=('pick_duration_min','median')).sort_values('avg_pick_time_min', ascending=False)
size_summary = clean.groupby('order_size_band', observed=False).agg(pick_count=('pick_id','count'), avg_pick_time_min=('pick_duration_min','mean'))
shift_summary = clean.groupby('shift').agg(pick_count=('pick_id','count'), avg_pick_time_min=('pick_duration_min','mean')).sort_values('avg_pick_time_min', ascending=False)
experience_summary = clean.groupby('experience_band').agg(pick_count=('pick_id','count'), avg_pick_time_min=('pick_duration_min','mean')).sort_values('avg_pick_time_min', ascending=False)
category_summary = clean.groupby('category').agg(pick_count=('pick_id','count'), avg_pick_time_min=('pick_duration_min','mean')).sort_values('avg_pick_time_min', ascending=False)
queue_summary = clean.groupby('priority').agg(order_count=('order_id','nunique'), avg_queue_min=('order_wait_min','mean'), median_queue_min=('order_wait_min','median')).sort_values('avg_queue_min', ascending=False)
display(zone_summary, size_summary, shift_summary, experience_summary, category_summary, queue_summary)


,pick_count,avg_pick_time_min,median_pick_time_min
zone,,,
C-02,387,13.650795,13.287212
C-01,380,13.480633,12.975535
B-01,391,12.630428,12.327713
B-02,361,12.378263,12.163555
A-02,383,11.771204,11.589636
A-01,375,11.732072,11.647387


,pick_count,avg_pick_time_min
order_size_band,,
Small (1-3),1609,12.636297
Medium (4-7),530,12.517504
Large (8-13),112,13.053603
Very large (14+),26,11.244780


,pick_count,avg_pick_time_min
shift,,
Night,575,15.443035
Unknown,66,13.423397
Evening,763,11.871109
Morning,873,11.336885


,pick_count,avg_pick_time_min
experience_band,,
Beginner,405,16.524337
Unknown,109,14.601002
Intermediate,1070,12.523403
Experienced,693,10.153736


,pick_count,avg_pick_time_min
category,,
Industrial,467,13.184721
Electronics,491,12.794360
Home,502,12.676647
Grocery,326,12.260639
Apparel,491,12.058057


,order_count,avg_queue_min,median_queue_min
priority,,,
High,244,35.644917,36.0
Low,147,34.848558,35.0
Normal,377,34.513353,34.0
Urgent,65,33.900524,33.0


## Feature engineering and model

In [4]:
features = ['quantity','item_count','priority','zone','zone_type','distance_band','category','size','storage_class','shift','experience_band','order_size_band','start_hour']
target = 'pick_duration_min'
X, y = clean[features], clean[target]
numeric = ['quantity','item_count','start_hour']
categorical = [c for c in features if c not in numeric]
prep = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical)
])
reg = Pipeline([('prep', prep), ('model', RandomForestRegressor(n_estimators=180, min_samples_leaf=3, random_state=22, n_jobs=-1))])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=22)
reg.fit(X_train, y_train)
pred = reg.predict(X_test)
mae = mean_absolute_error(y_test, pred)
rmse = math.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
print(f'Test MAE: {mae:.2f} minutes')
print(f'Test RMSE: {rmse:.2f} minutes')
print(f'Test R²: {r2:.3f}')


Test MAE: 2.04 minutes
Test RMSE: 2.54 minutes
Test R²: 0.563


## Feature importance and export

In [5]:
ohe = reg.named_steps['prep'].named_transformers_['cat'].named_steps['onehot']
feature_names = numeric + list(ohe.get_feature_names_out(categorical))
importance = pd.DataFrame({'feature': feature_names, 'importance': reg.named_steps['model'].feature_importances_}).sort_values('importance', ascending=False)
display(importance.head(15))
importance.to_csv(OUT/'feature_importance.csv', index=False)


,feature,importance
35,experience_band_Beginner,0.246481
0,quantity,0.210136
36,experience_band_Experienced,0.091781
33,shift_Night,0.084480
2,start_hour,0.061897
1,item_count,0.027581
32,shift_Morning,0.020285
13,zone_type_Bulk,0.018216
16,distance_band_Long,0.015752
38,experience_band_Unknown,0.011712
